In [1]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/Px_interface


In [2]:
%load_ext autoreload
%autoreload 2

import os, re, gc, ctypes, json, time, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import date
from rdkit import Chem
import yaml
import joblib
from tqdm import tqdm
from types import SimpleNamespace
tqdm.pandas()

# local modules
import python.functions as fn
import python.Px_interface as px
from get_library import get_df   # CDD Vault collection export

In [3]:
## reuse the CLI classes from python/Px_interface.py for debugging (%autoreload picks up edits)
from python.Px_interface import PARAMS, DATA, OUTPUT

CONFIG = 'config/config.yaml'   # swap to 'tmp/config.yaml' to debug on the synthetic fixture
OUTPUT_DIR = '/mnt/c/Users/gtamo/Desktop/GT/'           # base dir for the HTML + volcanoes (interfaces/ created under it)

## params:
params = PARAMS(CONFIG)
params.load_params()

In [4]:
## data:
data = DATA()
data.load_chemical_lib_df(params)
data.load_old_df(params)
data.load_new_df(params)
data.get_contaminants_and_controls(params)
data.get_gene_research(params)

> Chemical lib dim: (9827, 8)


/home/gtamo/miniconda3/envs/ML/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


> df_raw (24349157, 13) | MS (2277, 5)
> FBX: 8 tranches | MEASURE 20,016,464 rows | MSSCORE 22,842 | REPORT 2,412 (2,412 experiments)
> loaded degradation research for 10191 genes


In [5]:
## output:
output = OUTPUT()
output.combine_datasets(data, params)
output.get_de_validated(data, params)
output.get_iface(data, params)

> combined MEASURE: 43,565,325 rows | 4,821 experiments | 4,214 compounds | 12,062 genes
> combined MS-SCORE: 54,647 (gene,plate) rows (FBX 17,814, df_raw 36,833)
> combined REPORT: 4,821 uniquecontrasts | compounds 4,214 | plates 137
> plate dates: 151 plates mapped | report spans 2026-04-29 .. 2026-07-14
> target: 23 validated - 1124 devalidated targets
> compound: 39 validated - 138 devalidated compounds
> loaded interface inputs from output/interface/ | iface_df (8894, 5), compounds_df (64451, 9), meas (43565325, 11), 151 plate dates


In [6]:
## Build interface separately to avoid re-running all script
output.build_interface(data, params, OUTPUT_DIR)

> Plates default-ticked: 7 plate(s) on latest date 2026-07-14
> gene_research: 10191 genes
> 8,894 / 8,894 genes after filtering (x=R2, y=association_score, z=ms_score)
  [highlight] corner-top-40=40, association_score>None: 0, must=25, union=63
  [range_sliders] panels built for all 8894 plotted genes
> panels: loaded 8,894 gene panels from cache (skipped rebuild)
> no compound panel (provide compounds_df or top1..topN columns) — scatter + hover text only
  [range_sliders] default box ≈ 8894 genes; gene labels shown while ≤ 1000 in range (drag handles to widen/narrow)
  [activity_defaults] focus view starts on ['Low (2-10)', 'Single (1)']
  [gene_research] 7400/10191 records kept (plotted genes only) — 19.0 MB injected
wrote /mnt/c/Users/gtamo/Desktop/GT/interfaces/Serac_Px_interface.html  (2.6 MB main doc)  +  Serac_Px_interface_data.js  (34.1 MB, deferred)


In [26]:
# output.measure.to_parquet(params.GTLOCAL+'20260715_Px_MEASURE.parquet')
# output.mscore.to_parquet(params.GTLOCAL+'20260715_Px_MSCORE.parquet')
# output.report.to_parquet(params.GTLOCAL+'20260715_Px_REPORT.parquet')